# EnHi-LMP · ML-JudgeBERT with pretrained multilingual encoders (XLM-R, MuRIL, mBERT)
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Same protocol as the paper: 5-fold cross-validation grouped by clause (English/Hindi twins never split), 3 seeds, JUDGEBERT augmentation (identical → 10, unrelated → 1), MSE on the 1–10 LMP scale. Inside every training fold, 20 % of clauses are held out to pick the best epoch and the LegalDiff late-fusion weight.

| Step | Time on a T4 |
|---|---|
| Setup | ~2 min |
| XLM-R base | ~15 min |
| MuRIL base | ~15 min |
| mBERT (optional) | ~15 min |

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

In [ ]:
# 2. Get the code and install dependencies
import os
if not os.path.exists("NLP_RESERACH_WORK"):
    !git clone -q https://github.com/udbhav968-creator/NLP_RESERACH_WORK.git
%cd /content/NLP_RESERACH_WORK
!git log --oneline -3
!pip -q install sacrebleu wordllama openpyxl transformers accelerate sentencepiece

In [ ]:
# 3. Sanity check: LegalDiff demo (CPU, ~10 s)
!python src/demo.py

## 4. Pretrained encoders

In [ ]:
# 4a. XLM-RoBERTa base
!python src/finetune_pretrained.py --model xlm-roberta-base --epochs 8 --lr 2e-5

In [ ]:
# 4b. MuRIL base (trained on 17 Indian languages)
!python src/finetune_pretrained.py --model google/muril-base-cased --epochs 8 --lr 3e-5

In [ ]:
# 4c. (optional) multilingual BERT
!python src/finetune_pretrained.py --model bert-base-multilingual-cased --epochs 8 --lr 3e-5

## 5. Results

In [ ]:
# 5a. One table: CPU models from the paper (results/exp1_summary.json) + pretrained encoders
import json, glob, pandas as pd
rows = []
S = json.load(open("results/exp1_summary.json")); E1, BL = S["overall"], S["by_lang"]
for m, v in E1.items():
    a = v["all"]
    rows.append(dict(model=m, source="CPU (paper)", pearson=a["pearson"][0], spearman=a["spearman"][0], rmse=a["rmse"][0],
                     mae=a["mae"][0], f1_decision=a["f1_decision"][0], f1_notpreserved=a["f1_notpreserved"][0],
                     r_en=BL[m]["en"]["pearson"][0], r_hi=BL[m]["hi"]["pearson"][0]))
for f in sorted(glob.glob("results/pretrained_*.json")):
    if f.endswith("_predictions.json"): continue
    for m, v in json.load(open(f)).items():
        rows.append(dict(model=m, source="GPU (pretrained)", pearson=v["pearson"][0], spearman=v["spearman"][0], rmse=v["rmse"][0],
                         mae=v["mae"][0], f1_decision=v["f1_decision"][0], f1_notpreserved=v["f1_notpreserved"][0],
                         r_en=v["pearson_en"], r_hi=v["pearson_hi"], identical=v["identical_mean"], unrelated=v["unrelated_mean"]))
df = pd.DataFrame(rows).sort_values("pearson", ascending=False).round(3)
df.to_csv("results/all_models_comparison.csv", index=False)
df

In [ ]:
# 5b. Figure: Pearson r of every model, split by language
import matplotlib.pyplot as plt, numpy as np
d = df.dropna(subset=["r_en", "r_hi"]).sort_values("pearson")
y = np.arange(len(d)); h = 0.38
fig, ax = plt.subplots(figsize=(8, 0.45 * len(d) + 1.2))
ax.barh(y - h/2, d["r_en"], h, label="English", color="#4C72B0")
ax.barh(y + h/2, d["r_hi"], h, label="Hindi", color="#DD8452")
ax.set_yticks(y); ax.set_yticklabels(d["model"], fontsize=8); ax.set_xlabel("Pearson r with LMP labels")
ax.axvline(0, color="grey", lw=0.6); ax.legend(loc="lower right"); ax.set_title("EnHi-LMP: all models, by language")
plt.tight_layout(); plt.savefig("results/fig_pretrained_by_language.png", dpi=200); plt.show()

In [ ]:
# 5c. Download all results (JSON, CSV, figure) as one zip
!zip -qr enhi_lmp_gpu_results.zip results
from google.colab import files
files.download("enhi_lmp_gpu_results.zip")

## 6. After human annotation (final milestone)
Upload the two filled workbooks to `annotation/`, then run the cell below. It reports Krippendorff's α, lists pairs to adjudicate and re-runs the pretrained encoder on human labels only.

In [ ]:
# 6. Human labels: agreement + re-run (uncomment when both workbooks are filled)
# !python src/merge_annotations.py annotation/EnHi-LMP_annotation_A.xlsx annotation/EnHi-LMP_annotation_B.xlsx
# !LMP_DATA=enhi_lmp_human.jsonl python src/finetune_pretrained.py --model google/muril-base-cased --epochs 8 --lr 3e-5
# !LMP_DATA=enhi_lmp_human.jsonl python src/run_experiments.py exp1